# HW1 Question 1: Jegadeesh and Titman (1993)

Notebook scaffold for the momentum replication. Portfolio construction and inference are placeholders; the JT1993 downloader is drafted in `utils/jt1993_data_download.py`.

Use `references/jt1993_table1_wrds_replication.ipynb` as the construction benchmark. Keep sample choices, tables, graphs, and interpretation here. Reusable calculations belong in `utils/momentum.py` and `utils/inference.py`.

## Project layout

- `data/raw/jt1993/<universe>_<start>_<end>/`: cached WRDS extracts and query metadata.
- `data/processed/jt1993/`: future formation ranks, cohort returns, and strategy returns.
- `results/jt1993/tables/` and `results/jt1993/figures/`: final outputs.
- `tests/`: offline downloader checks and placeholders for portfolio/inference tests.

Run this notebook from the project root. Downloads are opt-in: the example below is commented out.

In [ ]:
from pathlib import Path

from utils.jt1993_data_download import DEFAULT_CACHE_DIR, download_jt1993_data

from utils.path_info import root_path, data_path, results_path

PROJECT_ROOT = Path(root_path)
PROCESSED_DIR = Path(data_path) / 'processed' / 'jt1993'
RESULT_DIR = Path(results_path) / 'jt1993'

# Raw-data dates are download arguments; reported return dates are analysis choices.
params = {
    'start': '1962-07-01',
    'end': '1989-12-31',
    'universe': 'nyse_amex',
}

settings = {
    'raw_dates': (params['start'], params['end']),
    'reported_dates': ('1965-01-31', '1989-12-31'),
    'universe': params['universe'],
    'exchange_codes': (1, 2),
    'share_codes': (10, 11),
    'formation_months': (3, 6, 9, 12),
    'holding_months': (3, 6, 9, 12),
    'panel_b_gap_trading_days': 5,
    'cache_root': str(DEFAULT_CACHE_DIR),
}
settings

## 1. Original-sample data

The dates below request July 1962-December 1989 for the original replication. Pass different raw dates through the downloader's start/end arguments; monthly datasets require complete calendar months. The universe argument currently supports only nyse_amex (NYSE/AMEX common shares) and reserves a place for future exchange samples. Other universes raise NotImplementedError. The reported portfolio-return sample starts in January 1965, after ranking and overlap initialization. Stocks are NYSE/AMEX common shares (exchange codes 1/2; share codes 10/11), using historical classifications valid on each observation date.

| Dataset | Source | Use |
|---|---|---|
| `daily_returns` | `crsp.dsf` and `crsp.msenames` | Panel B ranking signals and timing diagnostics |
| `monthly_returns` | Daily returns compounded on WRDS | Panel A signals and all monthly holding returns |
| `monthly_characteristics` | `crsp.msf` and `crsp.msenames` | Prices and shares for Table II market capitalization |
| `market_returns` | `crsp.msi` | CRSP value/equal-weighted indices; Table II market betas |

The downloader returns file paths and streams large queries to Parquet. Each file has a JSON sidecar with SQL, filters, row counts, coverage, and download time. It checks cached files before opening a WRDS connection.

Monthly return extracts are derived from daily data; `crsp.msf` returns do not replace that construction. Returns are decimals, monthly dates are calendar month ends, prices are dollars, and shares outstanding are thousands. Market capitalization in millions is `abs(prc) * shrout / 1000`. No separate delisting-return adjustment is applied. Missing daily returns remain missing in the daily extract; monthly compounding follows the reference notebook's valid-return rule.

Legacy reference caches remain unchanged. They are not automatically imported because their columns and provenance differ. Additional exchange universes and FF3 factor downloads will be added later.

### WRDS credentials

Pass your WRDS username explicitly, or set the `WRDS_USERNAME` user environment variable before starting the notebook kernel. Passwords belong in PostgreSQL's password file, outside this project. On Windows its default location is `%APPDATA%/postgresql/pgpass.conf`.

If a saved password is missing or outdated, WRDS may prompt once; accept its offer to create/update the password file after authentication. `db.create_pgpass_file()` is for one-time setup or updates, not every run. Duo approval can still be required.

See [WRDS's connection implementation](https://github.com/wharton/wrds/blob/main/wrds/sql.py), [PostgreSQL password files](https://www.postgresql.org/docs/current/libpq-pgpass.html), and [WRDS MFA guidance](https://wrds-www.wharton.upenn.edu/pages/about/log-in-to-wrds-using-two-factor-authentication/).

With valid caches, the downloader does not create a connection or ask for credentials. An owned connection is shared across missing downloads and closed afterward.

In [ ]:
# Uncomment when ready to download. Credentials can use WRDS_USERNAME.
# paths = download_jt1993_data(**params, username='YOUR_WRDS_USERNAME')

# Or request a subset, using WRDS_USERNAME:
# paths = download_jt1993_data(
#     **params, datasets=('monthly_returns', 'market_returns'),
# )

# Load only the needed dataset after download:
# import pandas as pd
# monthly = pd.read_parquet(paths['monthly_returns'])

## 2. Table I: relative-strength portfolio returns

TODO: implement Panel A and Panel B for J,K in {3,6,9,12}; retain formation membership, equal-weighted cohort returns, overlap counts, and monthly strategy returns. Compare means and t-statistics with the original reference outputs.

## 3. Table II: betas and market capitalization

TODO: keep all ten deciles for the (6,6) strategy; calculate post-ranking betas against the CRSP value-weighted index and average market capitalization. Document capitalization timing and units.

## 4. Table VII: performance in event time

TODO: follow six-month-ranked winner/loser cohorts for 36 event months. Report monthly and cumulative spread returns with the appropriate cumulative-return inference. Document endpoint censoring and the available cohort count at each horizon.

## 5. Later homework extensions

TODO: NYSE-only and NYSE/AMEX/Nasdaq samples, extended dates, two subsamples, a consolidated filter audit, and the FF3 regression for the (6,6) spread. Additional exchange universes and factor downloads are not implemented yet; raw date ranges can already be passed to the downloader.

## Offline verification

Run `.venv/Scripts/python.exe -B -m unittest discover -s tests -v` from the project root. Downloader tests use synthetic data and mocked connections; no WRDS login is required.